# Unit 3: Landslide Mapping with a Multi-Modal Vision Transformer

This notebook supports the Unit 3 ILOs:

- integrate spectral and terrain data into one feature set,
- apply a Vision Transformer that combines satellite and terrain features,
- analyze model limitations and misclassification patterns.

The model implementation is deliberately explicit so it can be studied and modified during teaching.

## Why This Notebook Uses Multi-Modal Learning

Landslide mapping benefits from combining satellite imagery with terrain information. Spectral data can reveal disturbed vegetation, exposed soil, and moisture-related change, while terrain features such as slope, elevation, and curvature help explain where landslides are physically plausible. This notebook is therefore designed to teach data fusion as well as modeling.

The Vision Transformer is used here to introduce a more advanced representation-learning approach. Instead of relying only on local convolutional filters, the model embeds image patches as tokens and learns relationships across the scene. That makes the notebook useful both for hazard mapping and for understanding modern multimodal model design.

## 1. Why a multi-modal model?

Landslides depend on both spectral evidence and topographic context. Satellite imagery may show bare soil, vegetation disturbance, or moisture contrasts, while terrain layers such as slope, elevation, and curvature explain where landslides are physically plausible.

A multi-modal Vision Transformer can fuse those sources into a common token representation.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path('data/unit3_landslide')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

In [ ]:
# Example external data hook.
# from datasets import load_dataset
# ds = load_dataset('your-org/landslide-multimodal-segmentation')

class PlaceholderLandslideDataset(Dataset):
    def __init__(self, n_samples=80, image_size=64):
        self.n_samples = n_samples
        self.image_size = image_size
        self.rng = np.random.default_rng(321)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        h = w = self.image_size
        spectral = self.rng.uniform(0, 1, size=(4, h, w)).astype(np.float32)
        y = np.linspace(0, 1, h, dtype=np.float32)
        x = np.linspace(0, 1, w, dtype=np.float32)
        yy, xx = np.meshgrid(y, x, indexing='ij')
        elevation = 0.5 * yy + 0.3 * np.sin(6 * xx)
        slope = np.abs(np.gradient(elevation, axis=0)) + np.abs(np.gradient(elevation, axis=1))
        curvature = np.gradient(np.gradient(elevation, axis=0), axis=0)
        terrain = np.stack([elevation, slope, curvature], axis=0).astype(np.float32)
        landslide_score = (slope > np.percentile(slope, 75)).astype(np.float32)
        landslide_score *= (spectral[3] < 0.45).astype(np.float32)
        mask = landslide_score[None, ...].astype(np.float32)
        return {
            'spectral': torch.from_numpy(spectral),
            'terrain': torch.from_numpy(terrain),
            'mask': torch.from_numpy(mask),
        }

train_ds = PlaceholderLandslideDataset(n_samples=96)
val_ds = PlaceholderLandslideDataset(n_samples=24)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        attn_input = self.norm1(x)
        attn_out, _ = self.attn(attn_input, attn_input, attn_input)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class MultiModalViTSegmenter(nn.Module):
    def __init__(self, spectral_channels=4, terrain_channels=3, image_size=64, patch_size=8, embed_dim=128, depth=4, num_heads=8):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.spectral_embed = PatchEmbed(spectral_channels, embed_dim, patch_size)
        self.terrain_embed = PatchEmbed(terrain_channels, embed_dim, patch_size)
        self.fusion = nn.Linear(embed_dim * 2, embed_dim)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerEncoderBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, embed_dim // 2, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(embed_dim // 2, embed_dim // 4, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(embed_dim // 4, embed_dim // 8, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim // 8, 1, kernel_size=1),
        )

    def forward(self, spectral, terrain):
        spectral_tokens = self.spectral_embed(spectral)
        terrain_tokens = self.terrain_embed(terrain)
        x = torch.cat([spectral_tokens, terrain_tokens], dim=-1)
        x = self.fusion(x)
        x = x + self.pos_embedding
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        b, n, c = x.shape
        x = x.transpose(1, 2).reshape(b, c, self.grid_size, self.grid_size)
        return self.decoder(x)

model = MultiModalViTSegmenter().to(DEVICE)
model

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def batch_dice(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (union + eps)).mean()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    for batch in loader:
        spectral = batch['spectral'].to(device)
        terrain = batch['terrain'].to(device)
        mask = batch['mask'].to(device)
        optimizer.zero_grad()
        logits = model(spectral, terrain)
        loss = criterion(logits, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * spectral.size(0)
        total_dice += batch_dice(logits.detach(), mask).item() * spectral.size(0)
    n = len(loader.dataset)
    return total_loss / n, total_dice / n

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    with torch.no_grad():
        for batch in loader:
            spectral = batch['spectral'].to(device)
            terrain = batch['terrain'].to(device)
            mask = batch['mask'].to(device)
            logits = model(spectral, terrain)
            loss = criterion(logits, mask)
            total_loss += loss.item() * spectral.size(0)
            total_dice += batch_dice(logits, mask).item() * spectral.size(0)
    n = len(loader.dataset)
    return total_loss / n, total_dice / n

In [ ]:
def confusion_counts(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).int()
    truth = targets.int()
    tp = int(((preds == 1) & (truth == 1)).sum().item())
    tn = int(((preds == 0) & (truth == 0)).sum().item())
    fp = int(((preds == 1) & (truth == 0)).sum().item())
    fn = int(((preds == 0) & (truth == 1)).sum().item())
    return {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

def inspect_failure_modes(batch, logits, threshold=0.5):
    counts = confusion_counts(logits, batch['mask'], threshold=threshold)
    print('Misclassification summary:', counts)
    print('Consider checking steep terrain, shadowed areas, roads, and exposed soil patches.')

## Discussion prompts

1. Why is terrain information especially important for landslides?
2. What is gained by tokenizing patches rather than using only convolutions?
3. How would incomplete landslide inventories bias validation scores?
4. Which false positives would you expect in mountainous terrain?

## Unit 3 Takeaway

The key lesson of this notebook is that landslide detection requires more than recognizing surface appearance. It benefits from combining spectral evidence with terrain structure. The multimodal transformer architecture reflects that idea directly by embedding each modality separately and learning from their fused representation.

Students should also recognize that model interpretation is essential. False positives and false negatives often reveal ambiguity in the landscape, incompleteness in the labels, or mismatch between model scale and geomorphic complexity. Understanding those failure modes is part of doing responsible GeoAI.